# 10 — IoU Threshold Sensitivity (DIAGNOSTIC)

> **This is a temporary diagnostic tool. It is NOT part of the final pipeline.**
> Do not use these results as primary outputs — they are sensitivity diagnostics only.

Examines how city-level F1 changes when the IoU match threshold varies.

**Current pipeline threshold:** `iou_threshold: 0.5` in `configs/validation_configs.yaml`.

**Limitation for τ = 0.25:**  
The pipeline's match parquets only store pairs that were matched at τ = 0.50.  
Near-miss pairs (0.25 ≤ IoU < 0.50) are not stored, so F1(τ=0.25) computed here is
a **lower bound** equal to F1(τ=0.50). To obtain the true value, re-run the pipeline
with `iou_threshold: 0.25` in `validation_configs.yaml`.

**Outputs:** `outputs/scratch/iou_threshold_sensitivity.csv`

In [ ]:
!pip install -q scikit-posthocs
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Cell 1 — Setup & load existing city-level results ─────────────────────────
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

CONFIG_PATH  = Path('/content/drive/MyDrive/WorldBank/FY26 - DEP/Gates Foundation/Building Dataset Validation/configs/validation_configs.yaml')
PROJECT_ROOT = CONFIG_PATH.parents[1]
sys.path.insert(0, str(PROJECT_ROOT))

METRICS_ROOT = PROJECT_ROOT / 'outputs' / 'metrics'
SCRATCH_DIR  = PROJECT_ROOT / 'outputs' / 'scratch'
GLOBAL_DIR   = PROJECT_ROOT / 'outputs' / 'global_metrics'
SCRATCH_DIR.mkdir(parents=True, exist_ok=True)

TILE_SENTINEL  = 'vector_metrics_tiles_all_datasets.parquet'
MATCH_SENTINEL = 'vector_matches_all_datasets.parquet'

# IoU threshold the pipeline was run with (= default 0.5)
TAU_PIPELINE = 0.50

# ── Load existing city-level summary (F1 at τ=0.5) ───────────────────────────
baseline_path = GLOBAL_DIR / 'vector_all_cities_merged.parquet'
if not baseline_path.exists():
    baseline_path = GLOBAL_DIR / 'vector_all_cities_merged.csv'
if baseline_path.exists():
    df_baseline = pd.read_parquet(baseline_path) if str(baseline_path).endswith('.parquet') else pd.read_csv(baseline_path)
    print(f'Baseline loaded: {len(df_baseline)} rows | {df_baseline["city"].nunique()} cities')
else:
    print('[WARN] vector_all_cities_merged not found — run notebook 04 first.')
    df_baseline = pd.DataFrame()

# ── Load SpaceNet7 flag from AOI tracker ─────────────────────────────────────
TRACKER_PATH = PROJECT_ROOT / 'data/02_interim/aoi_tracker.csv'
ref_source_map = {}
if TRACKER_PATH.exists():
    tracker = pd.read_csv(TRACKER_PATH, dtype=str)
    tracker.columns = tracker.columns.str.strip()
    if 'reference_source' in tracker.columns and 'dataset_folder_name' in tracker.columns:
        ref_source_map = (
            tracker.dropna(subset=['dataset_folder_name'])
            .set_index('dataset_folder_name')['reference_source']
            .str.strip().str.lower()
            .to_dict()
        )
        sn7 = sum(1 for v in ref_source_map.values() if v == 'spacenet')
        print(f'SpaceNet7 flag loaded: {len(ref_source_map)} cities, {sn7} SpaceNet')
    else:
        print('[WARN] reference_source column not in tracker — all cities treated as non-SpaceNet')
else:
    print(f'[WARN] Tracker not found at {TRACKER_PATH}')

print('\nCell 1 done.')

In [ ]:
# ── Cell 2 — Recompute F1 at τ = 0.25 and τ = 0.50 from stored matches ───────
#
# τ = 0.50 : exact  — uses stored tile metrics (source of truth)
# τ = 0.75 : exact  — filters stored matches to IoU ≥ 0.75 (stricter, fewer matches)
# τ = 0.25 : LOWER BOUND  — same as τ = 0.50 because stored matches all have
#            IoU ≥ 0.50; near-miss pairs (0.25 ≤ IoU < 0.50) are not stored.
#            True F1(τ=0.25) ≥ F1(τ=0.50). Re-run the pipeline with
#            iou_threshold: 0.25 in validation_configs.yaml for exact values.

THRESHOLDS = [0.25, 0.50, 0.75]  # 0.25 = lower bound; 0.50 = exact; 0.75 = exact


def city_f1_from_tile_metrics(tile_df: pd.DataFrame) -> pd.DataFrame:
    """Re-derive city-level F1 from sum of tile tp/fp/fn. Matches pipeline exactly."""
    rows = []
    for ds, g in tile_df.groupby('dataset'):
        tp = int(g['tp'].sum()); fp = int(g['fp'].sum()); fn = int(g['fn'].sum())
        p = tp / (tp + fp) if (tp + fp) else 0.0
        r = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = 2*p*r / (p+r) if (p+r) else 0.0
        rows.append({'dataset': ds, 'n_ref': int(g['n_ref'].sum()),
                     'n_cand': int(g['n_cand'].sum()), 'tp': tp, 'fp': fp, 'fn': fn,
                     'f1': round(f1, 4)})
    return pd.DataFrame(rows)


def city_f1_from_matches(tile_df: pd.DataFrame, matches_df: pd.DataFrame,
                         tau: float) -> pd.DataFrame:
    """
    City-level F1 by applying tau to stored match IoU values.
    n_ref and n_cand come from tile metrics (independent of threshold).
    tp = count of stored matches with IoU >= tau.
    NOTE: for tau < TAU_PIPELINE, this underestimates tp (lower bound).
    """
    rows = []
    for ds, g in tile_df.groupby('dataset'):
        n_ref  = int(g['n_ref'].sum())
        n_cand = int(g['n_cand'].sum())
        m_ds = (matches_df[(matches_df['dataset'] == ds) & (matches_df['iou'] >= tau)]
                if not matches_df.empty and 'dataset' in matches_df.columns
                else pd.DataFrame())
        tp = len(m_ds)
        fp = max(0, n_cand - tp)
        fn = max(0, n_ref  - tp)
        p  = tp / (tp + fp) if (tp + fp) else 0.0
        r  = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = 2*p*r / (p+r) if (p+r) else 0.0
        rows.append({'dataset': ds, 'tp_at_tau': tp, 'f1': round(f1, 4)})
    return pd.DataFrame(rows)


all_rows = []
missing_matches = []
city_dirs = sorted(p.parent for p in METRICS_ROOT.rglob(TILE_SENTINEL))
print(f'Processing {len(city_dirs)} cities...')

for city_dir in city_dirs:
    city = city_dir.name
    try:
        tile_df = pd.read_parquet(city_dir / TILE_SENTINEL)
    except Exception as e:
        print(f'  [WARN] {city}: tile metrics error: {e}')
        continue

    match_path = city_dir / MATCH_SENTINEL
    if match_path.exists():
        try:
            matches_df = pd.read_parquet(match_path)
        except Exception:
            matches_df = pd.DataFrame()
    else:
        matches_df = pd.DataFrame()
        missing_matches.append(city)

    # τ = 0.50 — exact from tile metrics
    f1_050 = city_f1_from_tile_metrics(tile_df).set_index('dataset')['f1'].to_dict()

    for ds in tile_df['dataset'].unique():
        row = {'city': city, 'dataset': ds,
               'is_spacenet7': (ref_source_map.get(city, 'other') == 'spacenet')}

        for tau in THRESHOLDS:
            if tau == TAU_PIPELINE:
                f1_val = f1_050.get(ds, float('nan'))
                is_lb  = False
            elif tau > TAU_PIPELINE:
                sub = city_f1_from_matches(tile_df, matches_df, tau)
                f1_val = sub.set_index('dataset')['f1'].get(ds, float('nan'))
                is_lb  = False
            else:  # tau < TAU_PIPELINE → lower bound
                # All stored matches have IoU ≥ 0.5 ≥ tau, so tp is identical to τ=0.5
                f1_val = f1_050.get(ds, float('nan'))
                is_lb  = True

            tau_key = f'f1_iou{int(tau*100):02d}'
            row[tau_key] = f1_val
            row[f'{tau_key}_is_lower_bound'] = is_lb

        all_rows.append(row)

df_sensitivity = pd.DataFrame(all_rows)
df_sensitivity['delta_iou25_vs_50'] = df_sensitivity['f1_iou25'] - df_sensitivity['f1_iou50']
df_sensitivity['delta_iou75_vs_50'] = df_sensitivity['f1_iou75'] - df_sensitivity['f1_iou50']

if missing_matches:
    print(f'\n[WARN] Match parquets missing for {len(missing_matches)} cities '
          f'(τ=0.75 set to NaN): {missing_matches[:5]}{"..." if len(missing_matches)>5 else ""}')

print(f'\nRecomputed: {len(df_sensitivity)} city×dataset rows')
print(f'  τ=0.25 column: LOWER BOUND (= τ=0.50 value, delta always 0 from stored data)')
print(f'  τ=0.50 column: exact')
print(f'  τ=0.75 column: exact (stricter — some matches dropped)')
print('\nSample (first 6 rows):')
display(df_sensitivity[['city', 'dataset', 'is_spacenet7',
                         'f1_iou25', 'f1_iou50', 'f1_iou75',
                         'delta_iou25_vs_50', 'delta_iou75_vs_50']].head(6))

In [ ]:
# ── Cell 3 — Comparison table + box plots ─────────────────────────────────────

# ── City-level summary (mean across datasets per city) ────────────────────────
df_city = (
    df_sensitivity
    .groupby(['city', 'is_spacenet7'])[['f1_iou25', 'f1_iou50', 'f1_iou75',
                                        'delta_iou25_vs_50', 'delta_iou75_vs_50']]
    .mean()
    .reset_index()
    .round(4)
)

print('=== City-level summary (mean F1 across datasets) ===')
print(f'Total cities: {len(df_city)}')
print(f'  SpaceNet: {df_city["is_spacenet7"].sum()} | Other: {(~df_city["is_spacenet7"]).sum()}')
print()

for group_label, grp in df_city.groupby('is_spacenet7'):
    label = 'SpaceNet7' if group_label else 'Non-SpaceNet'
    print(f'--- {label} ({len(grp)} cities) ---')
    for col in ['f1_iou25', 'f1_iou50', 'f1_iou75', 'delta_iou25_vs_50', 'delta_iou75_vs_50']:
        print(f'  {col:<30}: mean={grp[col].mean():.4f}  median={grp[col].median():.4f}  '
              f'std={grp[col].std():.4f}')
    print()

# ── Box plots ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

GROUP_PALETTE = {True: '#0072B2', False: '#E69F00'}
group_labels = {True: 'SpaceNet7', False: 'Non-SpaceNet'}

# Panel 1: F1 at different thresholds
df_melt = df_city.melt(
    id_vars=['city', 'is_spacenet7'],
    value_vars=['f1_iou25', 'f1_iou50', 'f1_iou75'],
    var_name='threshold', value_name='f1'
)
df_melt['group'] = df_melt['is_spacenet7'].map(group_labels)
df_melt['threshold_label'] = df_melt['threshold'].map(
    {'f1_iou25': 'τ=0.25 (LB)', 'f1_iou50': 'τ=0.50', 'f1_iou75': 'τ=0.75'})

sns.boxplot(data=df_melt, x='threshold_label', y='f1', hue='group',
            palette=GROUP_PALETTE, ax=axes[0], linewidth=1.2)
axes[0].set_title('F1 by threshold', fontweight='bold')
axes[0].set_xlabel('IoU threshold')
axes[0].set_ylabel('City-level F1')
axes[0].set_ylim(0, 1.05)
axes[0].legend(title='Group', fontsize=8)

# Panel 2: Delta τ=0.25 vs 0.50 (= 0 from stored data — shows the lower-bound issue)
df_city['group'] = df_city['is_spacenet7'].map(group_labels)
sns.boxplot(data=df_city, x='group', y='delta_iou25_vs_50',
            palette=GROUP_PALETTE, ax=axes[1], linewidth=1.2)
axes[1].axhline(0, color='grey', linestyle='--', linewidth=1)
axes[1].set_title('Δ F1: τ=0.25 minus τ=0.50\n(0 = lower bound; re-run for true Δ)',
                  fontweight='bold')
axes[1].set_xlabel('Group')
axes[1].set_ylabel('ΔF1 (τ0.25 − τ0.50)')

# Panel 3: Delta τ=0.75 vs 0.50 (real, from stored matches)
sns.boxplot(data=df_city, x='group', y='delta_iou75_vs_50',
            palette=GROUP_PALETTE, ax=axes[2], linewidth=1.2)
axes[2].axhline(0, color='grey', linestyle='--', linewidth=1)
axes[2].set_title('Δ F1: τ=0.75 minus τ=0.50\n(exact from stored matches)',
                  fontweight='bold')
axes[2].set_xlabel('Group')
axes[2].set_ylabel('ΔF1 (τ0.75 − τ0.50)')

for ax in axes:
    ax.grid(axis='y', alpha=0.3)
    sns.despine(ax=ax)

fig.suptitle('IoU threshold sensitivity — all datasets, by SpaceNet7 group',
             fontsize=13, fontweight='bold', y=1.01)
fig.tight_layout()
out_fig = SCRATCH_DIR / 'iou_threshold_sensitivity_boxplot.png'
fig.savefig(out_fig, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure saved → {out_fig}')

print('\n[NOTE] τ=0.25 delta is always 0 from stored match data (lower bound).')
print('       To obtain true Δ(τ=0.25 vs τ=0.50), re-run the pipeline with:')
print('         iou_threshold: 0.25  in configs/validation_configs.yaml')
print('       then re-run notebook 04 to regenerate vector_all_cities_merged,')
print('       and re-run this notebook.')

In [ ]:
# ── Cell 4 — Save output CSV ──────────────────────────────────────────────────
# Columns: city | dataset | is_spacenet7 | f1_iou50 | f1_iou25 | delta_iou25_vs_50
#          f1_iou75 | delta_iou75_vs_50 | f1_iou25_is_lower_bound

out_path = SCRATCH_DIR / 'iou_threshold_sensitivity.csv'

save_cols = [
    'city', 'dataset', 'is_spacenet7',
    'f1_iou50',
    'f1_iou25', 'f1_iou25_is_lower_bound', 'delta_iou25_vs_50',
    'f1_iou75', 'delta_iou75_vs_50',
]
save_cols = [c for c in save_cols if c in df_sensitivity.columns]
df_sensitivity[save_cols].to_csv(out_path, index=False)

print(f'Saved → {out_path}')
print(f'  Rows   : {len(df_sensitivity):,}')
print(f'  Columns: {save_cols}')
print()
print('=== Summary by group and dataset ===')
display(
    df_sensitivity
    .assign(group=df_sensitivity['is_spacenet7'].map({True: 'SpaceNet7', False: 'Non-SpaceNet'}))
    .groupby(['group', 'dataset'])[['f1_iou50', 'f1_iou75', 'delta_iou75_vs_50']]
    .agg(['mean', 'std'])
    .round(4)
)